# AF 8 runs -- dynamic + frame-by-frame ODLS (owner account)

Run this on the account that **owns** `MyDrive/odls_dynamic` (the one that holds the
preprocessed OCMR data and the AF 4 runs). No sharing or shortcuts needed.

It trains the two AF 8 runs **one after the other**, each fine-tuned from its AF 4 model:
1. dynamic ODLS (`KERNEL_T = 3`) -- ~2.5 h
2. frame-by-frame ODLS (`KERNEL_T = 1`) -- ~1.2 h

then evaluates both on the test set and compares all runs.

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then **Runtime -> Run all**.

**If the session ends early** (disconnect, GPU quota, 3 h time limit): open this notebook
again and **Run all**. Finished runs are skipped and an unfinished run resumes from the
exact batch where it stopped. When you stop training yourself, wait for
"Drive flushed -- everything is saved" before closing the tab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/odls_dynamic"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/ocmr_preprocessed"
LOCAL_DATA_DIR = "/content/ocmr_preprocessed"        # fast local copy, wiped with the runtime

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"
CODE_DIR = f"{REPO_DIR}/odls_dynamic"

AF = 8
SPARSE_DOMAIN = "yt"
EPOCHS = 20                # fine-tuning; raise and re-run Train if val is still improving
LR = 0.0005                # half the from-scratch rate, same for both runs (fair comparison)
BATCH_SIZE = 32
TIME_LIMIT_MIN = 180       # per run: stop cleanly + flush Drive after 3 h; re-run to continue

# The two AF 8 runs, in this order; each is fine-tuned from its AF 4 model's best.pt.
RUNS = [
    {"kernel_t": 3, "init_from": "dyn_af4_kt3_yt_v2", "label": "dynamic ODLS"},
    {"kernel_t": 1, "init_from": "dyn_af4_kt1_yt_v2", "label": "frame-by-frame ODLS"},
]


def run_dir(kernel_t):
    return f"{DRIVE_ROOT}/runs/dyn_af{AF:g}_kt{kernel_t}_{SPARSE_DOMAIN}_v2"

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
sys.path.insert(0, CODE_DIR)

from google.colab import drive


def ensure_drive():
    # If Drive is not mounted, /content/drive/... is just an empty LOCAL folder: checkpoints
    # written there vanish with the runtime. Train and Evaluate call this first.
    if not os.path.ismount('/content/drive'):
        print("Google Drive is not connected -- reconnecting (accept the pop-up if shown) ...")
        drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(DRIVE_DATA_DIR):
        raise RuntimeError(f"{DRIVE_DATA_DIR} not found -- is this the account that owns odls_dynamic?")


def flush_drive():
    # Blocks until everything written to Drive has finished uploading (files still queued
    # are LOST if the runtime is killed). Don't interrupt it -- let it print "Drive flushed".
    print("flushing Google Drive (waiting for uploads to finish -- don't stop this cell) ...")
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    print("Drive flushed -- everything is saved and Drive is connected again.")


!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Check the data and the AF 4 starting points (read-only)
Preprocessing was done once already; nothing is rewritten here.

In [ ]:
import csv
ensure_drive()
rows = list(csv.DictReader(open(f"{DRIVE_DATA_DIR}/manifest.csv")))
n_files = len([n for n in os.listdir(f"{DRIVE_DATA_DIR}/slices") if n.endswith(".npy")])
print(f"manifest: {len(rows)} slices, {n_files} slice files on Drive")
assert len(rows) == 128 and n_files == 128, "preprocessed data incomplete"
for run in RUNS:
    init = f"{DRIVE_ROOT}/runs/{run['init_from']}/best.pt"
    assert os.path.exists(init), f"missing AF 4 starting point: {init}"
    print(f"{run['label']:20s} starts from {init}")

## 2. Copy the data to local disk (every new session, ~1-4 min)
Reconnects Drive and carries on by itself if the connection drops mid-copy.

In [ ]:
import shutil, time


def remount():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)


src_dir, dst_dir = f"{DRIVE_DATA_DIR}/slices", f"{LOCAL_DATA_DIR}/slices"
os.makedirs(dst_dir, exist_ok=True)
for attempt in range(1, 6):
    try:
        names = sorted(n for n in os.listdir(src_dir) if n.endswith(".npy"))
        copied = 0
        for name in names:
            src, dst = f"{src_dir}/{name}", f"{dst_dir}/{name}"
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                continue
            shutil.copy(src, dst + ".tmp")
            os.replace(dst + ".tmp", dst)
            copied += 1
        for name in ("manifest.csv", "split.csv"):
            shutil.copy(f"{DRIVE_DATA_DIR}/{name}", f"{LOCAL_DATA_DIR}/{name}")
        print(f"done: copied {copied} new files, {len(names)} slices available locally")
        break
    except OSError as e:
        print(f"Drive connection dropped ({e}); reconnecting, attempt {attempt}/5 ...")
        time.sleep(5)
        remount()
else:
    raise RuntimeError("Drive kept disconnecting -- restart the runtime and run this cell again")

## 3. Train both AF 8 runs (one after the other)
Each run: `loaded weights from epoch ... best.pt`, then `START new run` (or `RESUME ...`
after a disconnect). A run that finished `EPOCHS` epochs is skipped; raise `EPOCHS` in
Config and re-run to train it further (it continues, it doesn't restart). If a run stops early
(3 h time limit, disconnect, or you press stop), Drive is flushed and the loop stops --
re-run this cell to continue that run, then the next one starts automatically.

In [ ]:
import torch


def finished(ckpt_dir):
    # Done = final.pt exists AND it reached EPOCHS (so raising EPOCHS continues the run).
    f = f"{ckpt_dir}/final.pt"
    return os.path.exists(f) and torch.load(f, map_location="cpu", weights_only=False).get("epoch", 0) >= EPOCHS


ensure_drive()
for run in RUNS:
    ckpt_dir = run_dir(run["kernel_t"])
    name = os.path.basename(ckpt_dir)
    if finished(ckpt_dir):
        print(f"{name}: already finished {EPOCHS} epochs -- skipping")
        continue
    init = f"{DRIVE_ROOT}/runs/{run['init_from']}/best.pt"
    cmd = (f"cd {CODE_DIR} && python -u train.py --data-dir {LOCAL_DATA_DIR} --checkpoint-dir {ckpt_dir} "
           f"--af {AF} --kernel-t {run['kernel_t']} --sparse-domain {SPARSE_DOMAIN} --epochs {EPOCHS} "
           f"--batch-size {BATCH_SIZE} --time-limit-min {TIME_LIMIT_MIN} --num-workers 2 "
           f"--init-checkpoint {init} --lr {LR}")
    print(f"\n===== {run['label']}: {name} =====")
    try:
        get_ipython().system(cmd)
    finally:
        flush_drive()
    if not finished(ckpt_dir):
        print(f"\n{name} stopped before finishing -- re-run this cell to continue it.")
        break
else:
    print("\nboth AF 8 runs finished")

## 4. Evaluate both on the test set
Uses each run's `best.pt`; AF 8 is taken from the checkpoint. Results go to
`<run>/eval_test_af8/results.json` on Drive.

In [ ]:
ensure_drive()
for run in RUNS:
    ckpt_dir = run_dir(run["kernel_t"])
    if not os.path.exists(f"{ckpt_dir}/best.pt"):
        print(f"{os.path.basename(ckpt_dir)}: no best.pt yet -- train it first")
        continue
    if not os.path.exists(f"{ckpt_dir}/final.pt"):
        print(f"NOTE: {os.path.basename(ckpt_dir)} hasn't finished training -- evaluating its best.pt so far")
    print(f"\n===== {run['label']} =====")
    get_ipython().system(f"cd {CODE_DIR} && python -u evaluate.py --data-dir {LOCAL_DATA_DIR} "
                         f"--checkpoint {ckpt_dir}/best.pt --split test")
flush_drive()

## 5. Compare all runs (AF 4 and AF 8)

In [ ]:
import glob, json
print(f"{'run':24s} {'AF':4s} {'method':22s} {'RLNE %':>7s} {'PSNR dB':>8s} {'SSIM %':>7s}")
for path in sorted(glob.glob(f"{DRIVE_ROOT}/runs/*/eval_test_*/results.json")):
    r = json.load(open(path))
    run = path.split("/runs/")[1].split("/")[0]
    kt = (r.get("model_config") or {}).get("kernel_t")
    for m, o in r["overall"].items():
        label = "zero-filled" if m == "zero_filled" else ("dynamic ODLS" if kt and kt > 1 else "frame-by-frame ODLS")
        print(f"{run:24s} {r['af_requested']:<4g} {label:22s} {o['RLNE_mean'] * 100:7.2f} "
              f"{o['PSNR_mean']:8.2f} {o['SSIM_mean'] * 100:7.2f}")

## 6. Side-by-side: zero-filled vs frame-by-frame vs dynamic at AF 8 (same slice, same mask)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
dirs = {run["label"]: f"{run_dir(run['kernel_t'])}/eval_test_af{AF:g}" for run in RUNS}
common = sorted(set.intersection(*[set(os.listdir(d)) for d in dirs.values() if os.path.isdir(d)] or [set()]))
common = [n for n in common if n.endswith(".npz")]
if len(dirs) != sum(os.path.isdir(d) for d in dirs.values()) or not common:
    print("evaluate both AF 8 runs first (step 4)")
else:
    name, t = common[0], None
    d0 = np.load(f"{dirs['dynamic ODLS']}/{name}")
    ref, zf = d0["ref"].astype(float), d0["zero_filled"].astype(float)
    t = ref.shape[0] // 2
    recs = {lab: np.load(f"{d}/{name}")["odls_dynamic"].astype(float) for lab, d in dirs.items()}
    panels = [("fully sampled", ref), (f"zero-filled AF={AF}", zf),
              ("frame-by-frame ODLS", recs["frame-by-frame ODLS"]), ("dynamic ODLS", recs["dynamic ODLS"])]
    col, vmax = ref.shape[2] // 2, ref.max()
    fig, ax = plt.subplots(3, 4, figsize=(16, 12))
    for j, (title, img) in enumerate(panels):
        ax[0, j].imshow(img[t], cmap="gray", vmin=0, vmax=vmax); ax[0, j].set_title(f"{title}, frame {t}")
        ax[1, j].imshow(np.abs(img[t] - ref[t]) * 5, cmap="gray", vmin=0, vmax=vmax); ax[1, j].set_title("|error| x5")
        ax[2, j].imshow(img[:, :, col].T, cmap="gray", vmin=0, vmax=vmax, aspect="auto"); ax[2, j].set_title("y-t profile")
    for a in ax.ravel():
        a.axis("off")
    plt.suptitle(f"{name}, AF {AF}"); plt.tight_layout(); plt.show()